# Tailscale 连接体检（单 cell，只读诊断）

Worker 连不上 hub 时跑这个：**它会逐层定位断在哪一层**，不改动任何训练状态。

| 层 | 看什么 | 常见结论 |
|---|---|---|
| 1 二进制 | `which tailscale` / version | 没装 ⇒ 引导会先装 |
| 2 守护进程·引擎 | socket / BackendState / `tailscaled` 日志尾 | kernel 模式 rc=1 ⇒ 应退 userspace |
| 3 登录 | Self IPv4 | `BackendState != Running` ⇒ authkey 用过/过期 |
| 4 peer | 对端 `online` / IP / lastSeen | 对端离线 ⇒ 本地 Tailscale 掉了 |
| 5 代理环境 | HTTP_PROXY / ALL_PROXY | userspace 模式必须靠它出站 |
| 6 hub 连通 | `curl /ping` 直连 + 走代理，各打 http_code | `000` ⇒ hub 绑 127.0.0.1 / 防火墙拦 8787 / 对端离线 |

跑完把输出整段贴出来即可定位。修逻辑去 `remote/tailscale_boot.py`（本 notebook 从 GitHub raw 拉它）。

**凭据**：留空，取用顺序 环境变量 → Colab/Kaggle Secrets → CFG 手填（`TS_AUTHKEY` / `HUB_TOKEN`）。


In [ ]:
# @title Tailscale 连接体检 —— 改参数后点 Run
import os, sys, time, urllib.request
from pathlib import Path

CFG = {
    "ts_authkey": "",             # TS_AUTHKEY（login=True 时用）
    "ts_ephemeral": True,
    "login": True,                # True=先起 daemon+登录再体检；False=只体检（daemon 会在第 2 层按需拉起）
    "hub_url": "http://<本地TS_IP>:8787",
    "hub_token": "",              # HUB_TOKEN（/ping 需要 Bearer）
    "repo_url": "https://github.com/HuangJian/battle.git",
    "branch": "goal-nn",
}


def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [ts-debug] {msg}", flush=True)


def _secret(key, cfg_val=""):
    """环境变量 → Colab/Kaggle Secrets → CFG 手填（值永不进日志）。"""
    def _env():
        return os.environ.get(key, "")

    def _colab():
        from google.colab import userdata
        return userdata.get(key) or ""

    def _kaggle():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key) or ""

    for _get in (_env, _colab, _kaggle):
        try:
            _v = _get()
        except Exception:
            _v = ""
        if _v:
            return str(_v).strip()
    return str(cfg_val or "").strip()


def _load(log):
    """拉 remote/tailscale_boot.py（本地有缓存先用）。"""
    dst_dir = Path("/tmp/battle-boot")
    dst = dst_dir / "tailscale_boot.py"
    base = CFG["repo_url"].replace("github.com", "raw.githubusercontent.com").rstrip("/")
    if base.endswith(".git"):
        base = base[:-4]
    _marker = dst_dir / "_branch.txt"
    if _marker.exists() and _marker.read_text(encoding="utf-8").strip() != CFG["branch"]:
        for _f in ("notebook_boot.py", "tailscale_boot.py"):
            (dst_dir / _f).unlink(missing_ok=True)
    dst_dir.mkdir(parents=True, exist_ok=True)
    _marker.write_text(CFG["branch"], encoding="utf-8")
    if not dst.exists():
        try:
            with urllib.request.urlopen(
                f"{base}/{CFG['branch']}/nn-training/remote/tailscale_boot.py",
                timeout=30) as _r:
                _data = _r.read()
            if b"def diagnose(" not in _data:
                raise ValueError("内容不像 tailscale_boot.py")
            dst_dir.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(_data)
        except Exception as _e:
            raise SystemExit(
                f"[FATAL] 拉不到引导模块（{type(_e).__name__}: {_e}）——检查 repo_url/branch，"
                f"或把仓库 push 到远端") from None
    sys.path.insert(0, str(dst_dir))
    try:
        import tailscale_boot
    except Exception as _e:
        raise SystemExit(f"[FATAL] 导入 tailscale_boot 失败: {type(_e).__name__}: {_e}") from None
    log(f"引导模块已载入（{CFG['branch']}）")
    return tailscale_boot


_boot = _load(_log)
if CFG.get("login"):
    try:
        _info = _boot.ensure({
            "ts_authkey": _secret("TS_AUTHKEY", CFG.get("ts_authkey")),
            "ts_ephemeral": bool(CFG.get("ts_ephemeral", True)),
            "proxy_env": True,
        }, _log)
        _log(f"登录完成: {_info}")
    except Exception as _e:
        _log(f"登录阶段失败: {type(_e).__name__}: {_e}（继续体检，问题大概率在第 2/3 层）")

_boot.diagnose({
    "hub_url": CFG.get("hub_url"),
    "hub_token": _secret("HUB_TOKEN", CFG.get("hub_token")),
}, _log)
